# Tahap 4: Case Solution Reuse (Prediksi Solusi)
**CBR - Pidana Umum: Pemalsuan**

Tujuan: Gunakan putusan lama (top-k hasil retrieval) sebagai dasar prediksi solusi untuk kasus baru.

**Algoritma:**
1. **Majority Vote**: Pilih solusi yang paling sering muncul di top-k
2. **Weighted Similarity**: Bobot = skor cosine similarity

**Output:** `predict_outcome(query)` → amar putusan yang diprediksi

## 1. Import & Load Semua Artefak dari Tahap Sebelumnya

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# Path
PROCESSED_DIR = Path("../data/processed")
EVAL_DIR      = Path("../data/eval")
RESULTS_DIR   = Path("../data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load kasus
with open(PROCESSED_DIR / "cases.json", encoding="utf-8") as f:
    cases = json.load(f)
case_lookup = {c["case_id"]: c for c in cases}

# Load solusi
with open(PROCESSED_DIR / "solutions.json", encoding="utf-8") as f:
    case_solutions = json.load(f)

# Load train embeddings
train_embeddings = np.load(EVAL_DIR / "train_embeddings.npy")
with open(EVAL_DIR / "train_case_ids.json") as f:
    train_case_ids = json.load(f)

# Load split info
with open(EVAL_DIR / "data_split.json") as f:
    split_info = json.load(f)

# Load queries
with open(EVAL_DIR / "queries.json", encoding="utf-8") as f:
    test_queries = json.load(f)

print(f"✅ Data dimuat: {len(cases)} kasus, {len(train_case_ids)} train, {len(test_queries)} query")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

✅ Data dimuat: 35 kasus, 28 train, 7 query
🖥️  Device: cpu


## 2. Load Model IndoBERT (untuk embed query baru)

In [2]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 512

print(f"⏳ Memuat model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("✅ Model siap!")

def embed_text(text: str) -> np.ndarray:
    """Embed satu teks dengan IndoBERT (mean pooling + L2 norm)."""
    encoded = tokenizer(
        text, max_length=MAX_LENGTH,
        padding=True, truncation=True, return_tensors="pt"
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model(**encoded)
    token_emb = output[0]
    mask = encoded["attention_mask"].unsqueeze(-1).expand(token_emb.size()).float()
    mean_emb = torch.sum(token_emb * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
    mean_emb = torch.nn.functional.normalize(mean_emb, p=2, dim=1)
    return mean_emb.cpu().numpy()

⏳ Memuat model: indobenchmark/indobert-base-p1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model siap!


## 3. Fungsi Retrieve (ulang dari Tahap 3)

In [3]:
def retrieve(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus mirip dari case base.
    Returns: list of (case_id, similarity_score)
    """
    query_emb  = embed_text(query.lower().strip())
    sims       = cosine_similarity(query_emb, train_embeddings)[0]
    top_k_idx  = np.argsort(sims)[::-1][:k]
    return [(train_case_ids[i], float(sims[i])) for i in top_k_idx]

print("✅ Fungsi retrieve() siap")

✅ Fungsi retrieve() siap


## 4. Algoritma Prediksi Solusi

In [4]:
def normalize_vonis(vonis_text: str) -> str:
    """
    Normalisasi teks vonis untuk perbandingan.
    Ekstrak durasi penjara sebagai label kategori.
    """
    import re
    if not vonis_text or vonis_text in ("TIDAK_DITEMUKAN", "TIDAK_DIKETAHUI"):
        return "TIDAK_DIKETAHUI"
    
    text = vonis_text.lower()
    
    # Cek bebas
    if any(k in text for k in ['bebas', 'lepas dari']):
        return "BEBAS"
    
    # Ekstrak durasi
    match = re.search(r'(\d+)\s*tahun', text)
    if match:
        years = int(match.group(1))
        return f"PENJARA_{years}_TAHUN"
    
    match = re.search(r'(\d+)\s*bulan', text)
    if match:
        months = int(match.group(1))
        return f"PENJARA_{months}_BULAN"
    
    return "PENJARA_LAIN"


def predict_outcome(
    query: str,
    k: int = 5,
    method: str = "weighted"  # 'majority' atau 'weighted'
) -> dict:
    """
    Prediksi solusi (amar putusan) untuk kasus baru.
    
    Args:
        query  : Deskripsi kasus baru
        k      : Jumlah kasus teratas untuk konsultasi
        method : 'majority' (voting) atau 'weighted' (bobot similarity)
    
    Returns:
        Dict berisi prediksi dan metadata
    """
    # Step 1: Retrieve top-k kasus
    top_k = retrieve(query, k=k)  # [(case_id, score), ...]
    top_k_ids    = [t[0] for t in top_k]
    top_k_scores = [t[1] for t in top_k]
    
    # Step 2: Kumpulkan solusi
    solutions = []
    for cid, score in top_k:
        amar   = case_solutions.get(cid, "TIDAK_DITEMUKAN")
        vonis  = case_lookup.get(cid, {}).get("vonis", "TIDAK_DITEMUKAN")
        vonis_norm = normalize_vonis(vonis)
        solutions.append({
            "case_id"   : cid,
            "similarity": score,
            "amar"      : amar,
            "vonis"     : vonis,
            "vonis_norm": vonis_norm
        })
    
    # Step 3: Pilih solusi
    if method == "majority":
        # Majority Vote: hitung frekuensi vonis
        vonis_counts = Counter([s["vonis_norm"] for s in solutions])
        predicted_label = vonis_counts.most_common(1)[0][0]
        # Ambil amar dari kasus dengan vonis terpilih (yang pertama)
        predicted_amar = next(
            (s["amar"] for s in solutions if s["vonis_norm"] == predicted_label),
            solutions[0]["amar"]
        )
        
    else:  # weighted
        # Weighted Similarity: bobot = skor similarity
        # Hitung skor tertimbang per label vonis
        weighted_scores = {}
        for s in solutions:
            lbl = s["vonis_norm"]
            weighted_scores[lbl] = weighted_scores.get(lbl, 0) + s["similarity"]
        
        predicted_label = max(weighted_scores, key=weighted_scores.get)
        predicted_amar = next(
            (s["amar"] for s in solutions if s["vonis_norm"] == predicted_label),
            solutions[0]["amar"]
        )
    
    return {
        "predicted_vonis"   : predicted_label,
        "predicted_solution": predicted_amar[:400],
        "top_k_case_ids"    : top_k_ids,
        "top_k_scores"      : top_k_scores,
        "top_k_solutions"   : solutions,
        "method"            : method
    }


print("✅ Fungsi predict_outcome() siap!")

✅ Fungsi predict_outcome() siap!


## 5. Demo Manual: Prediksi 5 Kasus Baru

In [5]:
# Demo prediksi menggunakan kasus test sebagai 'kasus baru'
df = pd.DataFrame(cases)
df_test  = df[df["case_id"].isin(split_info["test_ids"])].reset_index(drop=True)

print("🎯 Demo Prediksi Solusi (5 Kasus Test Pertama)")
print("="*70)

demo_results = []
for i, row in df_test.head(5).iterrows():
    query_text = f"{row.get('dakwaan', '')} {row.get('ringkasan_fakta', '')}"
    actual_vonis = row.get("vonis", "TIDAK_DIKETAHUI")
    actual_vonis_norm = normalize_vonis(actual_vonis)
    
    # Prediksi dengan kedua metode
    pred_majority = predict_outcome(query_text, k=5, method="majority")
    pred_weighted = predict_outcome(query_text, k=5, method="weighted")
    
    print(f"\n[Kasus {i+1}] {row['case_id']} | {row['no_perkara']}")
    print(f"  Pasal        : {row.get('pasal', '-')}")
    print(f"  Aktual Vonis : {actual_vonis_norm}")
    print(f"  Pred Majority: {pred_majority['predicted_vonis']}")
    print(f"  Pred Weighted: {pred_weighted['predicted_vonis']}")
    print(f"  Top-5 Cases  : {pred_majority['top_k_case_ids']}")
    print(f"  Top-5 Scores : {[f'{s:.3f}' for s in pred_majority['top_k_scores']]}")
    
    demo_results.append({
        "query_id"           : row["case_id"],
        "no_perkara"         : row["no_perkara"],
        "actual_vonis"       : actual_vonis_norm,
        "predicted_majority" : pred_majority["predicted_vonis"],
        "predicted_weighted" : pred_weighted["predicted_vonis"],
        "predicted_solution" : pred_weighted["predicted_solution"],
        "top_5_case_ids"     : ", ".join(pred_majority["top_k_case_ids"]),
        "top_5_scores"       : ", ".join([f"{s:.4f}" for s in pred_majority["top_k_scores"]])
    })

🎯 Demo Prediksi Solusi (5 Kasus Test Pertama)

[Kasus 1] case_014 | 191/PID.B/2012/PN.TRK
  Pasal        : 199, 263, 263 2, 199 1, 197 1
  Aktual Vonis : PENJARA_LAIN
  Pred Majority: PENJARA_LAIN
  Pred Weighted: PENJARA_LAIN
  Top-5 Cases  : ['case_033', 'case_001', 'case_002', 'case_011', 'case_008']
  Top-5 Scores : ['0.795', '0.792', '0.781', '0.780', '0.778']



[Kasus 2] case_016 | 149/PID.B/2012/PN.BJB
  Pasal        : 253, 263, 385, 406, 263 1
  Aktual Vonis : PENJARA_LAIN
  Pred Majority: PENJARA_LAIN
  Pred Weighted: PENJARA_LAIN
  Top-5 Cases  : ['case_029', 'case_034', 'case_033', 'case_031', 'case_021']
  Top-5 Scores : ['0.889', '0.845', '0.839', '0.834', '0.824']

[Kasus 3] case_020 | 327 K/PID/2017
  Pasal        : 52, 253, 263 1, 188 1, 191 2
  Aktual Vonis : PENJARA_LAIN
  Pred Majority: PENJARA_LAIN
  Pred Weighted: PENJARA_LAIN
  Top-5 Cases  : ['case_002', 'case_010', 'case_012', 'case_028', 'case_021']
  Top-5 Scores : ['0.843', '0.830', '0.822', '0.817', '0.816']



[Kasus 4] case_022 | 35 K/PID/2015
  Pasal        : 378, 266 2, 55 1, 263 1, 263 2
  Aktual Vonis : TIDAK_DIKETAHUI
  Pred Majority: PENJARA_LAIN
  Pred Weighted: PENJARA_LAIN
  Top-5 Cases  : ['case_010', 'case_002', 'case_033', 'case_011', 'case_001']
  Top-5 Scores : ['0.836', '0.833', '0.831', '0.829', '0.825']


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]


[Kasus 5] case_025 | 2418/PID.B/2005/PN.SBY
  Pasal        : 372, 368, 37 1, 87 1, 363 1
  Aktual Vonis : PENJARA_LAIN
  Pred Majority: PENJARA_LAIN
  Pred Weighted: PENJARA_LAIN
  Top-5 Cases  : ['case_031', 'case_029', 'case_019', 'case_018', 'case_010']
  Top-5 Scores : ['0.780', '0.767', '0.749', '0.749', '0.744']


## 6. Simpan Hasil Prediksi

In [6]:
# Jalankan prediksi untuk semua kasus test
all_predictions = []

print("⏳ Memprediksi semua kasus test...")
for _, row in df_test.iterrows():
    query_text = f"{row.get('dakwaan', '')} {row.get('ringkasan_fakta', '')}"
    actual_vonis_norm = normalize_vonis(row.get("vonis", "TIDAK_DIKETAHUI"))
    
    pred = predict_outcome(query_text, k=5, method="weighted")
    
    all_predictions.append({
        "query_id"           : row["case_id"],
        "no_perkara"         : row.get("no_perkara", "-"),
        "actual_vonis"       : actual_vonis_norm,
        "predicted_solution" : pred["predicted_solution"],
        "predicted_vonis"    : pred["predicted_vonis"],
        "top_5_case_ids"     : ", ".join(pred["top_k_case_ids"]),
        "top_5_scores"       : ", ".join([f"{s:.4f}" for s in pred["top_k_scores"]])
    })

# Simpan ke CSV
df_pred = pd.DataFrame(all_predictions)
pred_path = RESULTS_DIR / "predictions.csv"
df_pred.to_csv(pred_path, index=False, encoding="utf-8-sig")

print(f"\n✅ {len(all_predictions)} prediksi disimpan di: {pred_path}")
print(f"\nPreview:")
print(df_pred[["query_id", "actual_vonis", "predicted_vonis", "top_5_case_ids"]].to_string(index=False))

print("\n✅ Tahap 4 selesai! Lanjutkan ke notebook 05_evaluation.ipynb")

⏳ Memprediksi semua kasus test...



✅ 7 prediksi disimpan di: ..\data\results\predictions.csv

Preview:
query_id    actual_vonis predicted_vonis                                   top_5_case_ids
case_014    PENJARA_LAIN    PENJARA_LAIN case_033, case_001, case_002, case_011, case_008
case_016    PENJARA_LAIN    PENJARA_LAIN case_029, case_034, case_033, case_031, case_021
case_020    PENJARA_LAIN    PENJARA_LAIN case_002, case_010, case_012, case_028, case_021
case_022 TIDAK_DIKETAHUI    PENJARA_LAIN case_010, case_002, case_033, case_011, case_001
case_025    PENJARA_LAIN    PENJARA_LAIN case_031, case_029, case_019, case_018, case_010
case_027    PENJARA_LAIN           BEBAS case_006, case_035, case_024, case_029, case_004
case_030    PENJARA_LAIN    PENJARA_LAIN case_033, case_029, case_006, case_032, case_009

✅ Tahap 4 selesai! Lanjutkan ke notebook 05_evaluation.ipynb
